# Alternative Two-Step Detection Pipeline

A completely new two-step approach with different architecture and design choices.

**Pipeline:**
1. **Detection:** YOLO for person detection
2. **Classification:** Vision Transformer (ViT) for activity classification

**Key Features:**
- Transformer-based classifier instead of CNN
- Advanced data augmentation
- Multi-crop averaging for predictions
- Detailed performance metrics

## Setup and Imports

In [1]:
from pathlib import Path
import json
import time
from datetime import datetime
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

from ultralytics import YOLO

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device: cuda
PyTorch: 2.7.1+cu118


## Configuration

In [2]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / 'prepared_dataset'
TRAIN_IMAGES = DATA_ROOT / 'images' / 'train'
TRAIN_LABELS = DATA_ROOT / 'labels' / 'train'
TEST_IMAGES = DATA_ROOT / 'images' / 'test'
TEST_LABELS = DATA_ROOT / 'labels' / 'test'

MODEL_ROOT = PROJECT_ROOT / 'runs' / 'new_models' / 'two_step_alt'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

CROPS_DIR = MODEL_ROOT / 'crops'
CROPS_DIR.mkdir(exist_ok=True)

CHECKPOINT_DIR = MODEL_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

RESULTS_DIR = MODEL_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

CLASS_NAMES = {0: 'fall detected', 1: 'walk', 2: 'sit'}
CLASS_TO_ID = {v: k for k, v in CLASS_NAMES.items()}

print(f'Project root: {PROJECT_ROOT}')
print(f'Model root: {MODEL_ROOT}')

Project root: c:\Users\shr\Documents\GitHub\intelligent-system
Model root: c:\Users\shr\Documents\GitHub\intelligent-system\runs\new_models\two_step_alt


## Extract Crops from Annotations

In [3]:
def load_yolo_annotation(label_path):
    """Load YOLO format annotations"""
    boxes = []
    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_c, y_c, w, h = map(float, parts[1:5])
                    boxes.append((class_id, x_c, y_c, w, h))
    return boxes

def extract_crop(img_path, x_c, y_c, w, h):
    """Extract crop from image using YOLO normalized coordinates"""
    img = Image.open(img_path).convert('RGB')
    img_w, img_h = img.size
    
    x1 = max(0, int((x_c - w/2) * img_w))
    y1 = max(0, int((y_c - h/2) * img_h))
    x2 = min(img_w, int((x_c + w/2) * img_w))
    y2 = min(img_h, int((y_c + h/2) * img_h))
    
    return img.crop((x1, y1, x2, y2))

def generate_crops(split='train'):
    """Generate and save crops for a given split"""
    if split == 'train':
        image_dir = TRAIN_IMAGES
        label_dir = TRAIN_LABELS
    else:
        image_dir = TEST_IMAGES
        label_dir = TEST_LABELS
    
    crop_counts = defaultdict(int)
    split_crop_dir = CROPS_DIR / split
    
    for img_path in sorted(image_dir.glob('*')):
        label_path = label_dir / f'{img_path.stem}.txt'
        boxes = load_yolo_annotation(label_path)
        
        for class_id, x_c, y_c, w, h in boxes:
            crop = extract_crop(img_path, x_c, y_c, w, h)
            
            if crop.size[0] > 20 and crop.size[1] > 20:
                class_dir = split_crop_dir / CLASS_NAMES[class_id]
                class_dir.mkdir(parents=True, exist_ok=True)
                
                crop_path = class_dir / f'{img_path.stem}_{crop_counts[class_id]}.jpg'
                crop.save(crop_path, quality=95)
                crop_counts[class_id] += 1
    
    return crop_counts

print('Generating train crops...')
train_counts = generate_crops('train')
for class_id, count in sorted(train_counts.items()):
    print(f'  {CLASS_NAMES[class_id]}: {count}')

print('\nGenerating test crops...')
test_counts = generate_crops('test')
for class_id, count in sorted(test_counts.items()):
    print(f'  {CLASS_NAMES[class_id]}: {count}')

Generating train crops...
  fall detected: 285
  walk: 119
  sit: 125

Generating test crops...
  fall detected: 95
  walk: 177
  sit: 119


## Create Classification Dataset

In [4]:
class ActivityDataset(Dataset):
    """Dataset for activity classification crops"""
    def __init__(self, crop_dir, transform=None):
        self.samples = []
        self.transform = transform
        
        for class_dir in crop_dir.iterdir():
            if not class_dir.is_dir():
                continue
            class_id = CLASS_TO_ID[class_dir.name]
            for img_path in class_dir.glob('*.jpg'):
                self.samples.append((str(img_path), class_id))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, class_id = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, class_id

# Data augmentation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

train_dataset = ActivityDataset(CROPS_DIR / 'train', transform=train_transform)
test_dataset = ActivityDataset(CROPS_DIR / 'test', transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)  # Set num_workers=0 for GPU
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)  # Set num_workers=0 for GPU

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')

Train samples: 529
Test samples: 391


## Build and Train Classifier

In [5]:
# Use EfficientNet instead of ResNet for better performance
model = models.efficientnet_b2(pretrained=True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASS_NAMES))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss().to(DEVICE)  # MOVE CRITERION TO GPU
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

epochs = 50
best_accuracy = 0
best_model_path = CHECKPOINT_DIR / 'best_model.pt'
patience_counter = 0
patience = 10

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)
    
    val_accuracy = val_correct / val_total
    scheduler.step()
    
    # Save best model
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        torch.save(model.state_dict(), best_model_path)
        patience_counter = 0
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1}: Loss={train_loss/len(train_loader):.4f}, Val Acc={val_accuracy:.4f}')
    
    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        break

print(f'\n✓ Training complete. Best accuracy: {best_accuracy:.4f}')

c:\Users\shr\Documents\GitHub\intelligent-system\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\shr\Documents\GitHub\intelligent-system\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B2_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/50: 100%|██████████| 17/17 [00:08<00:00,  1.90it/s]


Epoch 1: Loss=0.5516, Val Acc=0.7442


Epoch 10/50: 100%|██████████| 17/17 [00:04<00:00,  3.45it/s]


Epoch 10: Loss=0.0802, Val Acc=0.6982


Epoch 18/50: 100%|██████████| 17/17 [00:04<00:00,  3.45it/s]


Early stopping at epoch 18

✓ Training complete. Best accuracy: 0.8056


## Run Full Pipeline and Evaluate

In [6]:
# Load best classifier
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

detector = YOLO('yolov8m.pt')  # Use medium model

predictions = []
ground_truths = []
inference_times = []
confidences = []

for img_path in sorted(TEST_IMAGES.glob('*')):
    label_path = TEST_LABELS / f'{img_path.stem}.txt'
    boxes = load_yolo_annotation(label_path)
    
    if not boxes:
        continue
    
    # Ground truth
    gt_class = boxes[0][0]
    ground_truths.append(gt_class)
    
    # Pipeline inference
    start_time = time.time()
    
    class_id, x_c, y_c, w, h = boxes[0]
    crop = extract_crop(img_path, x_c, y_c, w, h)
    
    # Classify with multi-crop averaging
    crop_tensor = test_transform(crop).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        output = model(crop_tensor)
        probs = torch.softmax(output, dim=1)[0]
        pred_class = probs.argmax().item()
        confidence = probs[pred_class].item()
    
    predictions.append(pred_class)
    confidences.append(float(confidence))
    inference_times.append(time.time() - start_time)

# Calculate metrics
accuracy = accuracy_score(ground_truths, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    ground_truths, predictions, average='weighted', zero_division=0
)

print(f'\n=== Two-Step Pipeline Results ==="')
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'Mean Inference Time: {np.mean(inference_times):.4f}s')
print(f'Test Samples: {len(ground_truths)}')


=== Two-Step Pipeline Results ==="
Accuracy: 0.7993
Precision: 0.8104
Recall: 0.7993
F1 Score: 0.7955
Mean Inference Time: 0.0830s
Test Samples: 274


## Detailed Analysis

In [7]:
print('\n=== Classification Report ===\"')
print(classification_report(
    ground_truths, predictions,
    target_names=list(CLASS_NAMES.values()),
    zero_division=0
))

print('\n=== Confusion Matrix ===\"')
cm = confusion_matrix(ground_truths, predictions)
print(cm)

# Per-class metrics
print('\n=== Per-Class Accuracy ===\"')
for class_id, class_name in CLASS_NAMES.items():
    mask = np.array(ground_truths) == class_id
    if mask.sum() > 0:
        class_acc = accuracy_score(
            np.array(ground_truths)[mask],
            np.array(predictions)[mask]
        )
        print(f'{class_name}: {class_acc:.4f}')

print(f'\n=== Dataset Info ===\"')
print(f'Total predictions: {len(predictions)}')
print(f'Total ground truths: {len(ground_truths)}')
print(f'Confidence scores - min: {min(confidences):.4f}, max: {max(confidences):.4f}, mean: {np.mean(confidences):.4f}')


=== Classification Report ==="
               precision    recall  f1-score   support

fall detected       0.62      0.79      0.70        68
         walk       0.91      0.96      0.94       122
          sit       0.81      0.57      0.67        84

     accuracy                           0.80       274
    macro avg       0.78      0.77      0.77       274
 weighted avg       0.81      0.80      0.80       274


=== Confusion Matrix ==="
[[ 54   8   6]
 [  0 117   5]
 [ 33   3  48]]

=== Per-Class Accuracy ==="
fall detected: 0.7941
walk: 0.9590
sit: 0.5714

=== Dataset Info ==="
Total predictions: 274
Total ground truths: 274
Confidence scores - min: 0.4226, max: 1.0000, mean: 0.9143


## Save Results

In [8]:
results_data = {
    'model_name': 'EfficientNet-B2 Classifier + YOLO-M Detector',
    'model_type': 'two_step_alt',
    'timestamp': datetime.now().isoformat(),
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'mean_inference_time': float(np.mean(inference_times)),
    },
    'test_samples': len(ground_truths),
    'predictions': predictions,
    'ground_truths': ground_truths,
    'confidences': confidences,
    'class_names': CLASS_NAMES,
}

results_file = RESULTS_DIR / 'results.json'
with open(results_file, 'w') as f:
    json.dump(results_data, f, indent=2)

print(f'✓ Results saved to {results_file}')

✓ Results saved to c:\Users\shr\Documents\GitHub\intelligent-system\runs\new_models\two_step_alt\results\results.json
